# Imports và Cấu hình (Configuration)

In [ ]:
import os
import math
import random
from collections import deque

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

# Custom imports
from Bert4Rec_model import BERT4Rec, gather_indexes

# Cấu hình môi trường và đường dẫn
os.environ['CUDA_LAUNCH_BLOCKING'] = '1' 
os.environ['TORCH_USE_CUDA_DSA'] = '1'

RATING_FILE_PATH = r'F:\1_REL\Reinforcement-learning-for-Recommendation\Data_Movielens_1m\ml-1m\ratings.dat'
MODEL_WEIGHT_PATH = r'F:\1_REL\Reinforcement-learning-for-Recommendation\saved_models\bert4rec_best.pth'
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

**seed**

chỉ chạy 1 lần

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # Nếu dùng nhiều GPU
    
    # Đảm bảo tính toán trên GPU nhất quán (có thể làm chậm một chút)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Áp dụng
set_seed(42)

# Data Processing & Dataloader

In [ ]:
def load_offline_data_custom(rating_file_path, max_history_len=200, min_history_len=5):
    print(f"[*] Đang tải dữ liệu từ {rating_file_path} ...")
    ratings_cols = ['UserID', 'MovieID', 'Rating', 'Timestamp']
    ratings = pd.read_csv(rating_file_path, sep='::', engine='python', names=ratings_cols)
    ratings = ratings.sort_values(by=['UserID', 'Timestamp']).reset_index(drop=True)

    raw_movie_ids = sorted(ratings['MovieID'].unique())
    movie2id = {raw_id: i + 1 for i, raw_id in enumerate(raw_movie_ids)}
    ratings['Movie_Encoded'] = ratings['MovieID'].map(movie2id)

    print("[*] Đang gom nhóm thành lịch sử Sequence...")
    ratings['pair'] = list(zip(ratings['Movie_Encoded'], ratings['Rating']))
    user_sequences = ratings.groupby('UserID')['pair'].apply(list).to_dict()

    offline_data = []
    for _, seq in user_sequences.items():
        if len(seq) < 3: continue
        
        last_valid_i = None
        for i in range(len(seq) - 1, 0, -1):
            if len([m for m, r in seq[max(0, i - max_history_len):i]]) >= min_history_len:
                last_valid_i = i
                break

        for i in range(1, len(seq)):
            history = [m for m, r in seq[max(0, i - max_history_len):i]]
            if len(history) < min_history_len: continue
            
            target_item = seq[i][0]
            target_rating = seq[i][1]
            is_terminal = (i == last_valid_i)
            offline_data.append((history, target_item, target_rating, is_terminal))

    print(f"[+] Hoàn tất! Đã tạo được {len(offline_data):,} mẫu Transition.")
    return offline_data

class OfflineRLDataset(Dataset):
    def __init__(self, offline_data):
        self.data = offline_data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def collate_fn(batch):
    histories, target_items, target_ratings, is_terminals = zip(*batch)
    max_len = max(len(h) for h in histories)
    padded  = [h + [0] * (max_len - len(h)) for h in histories]
    
    return (
        padded, 
        list(target_items), 
        torch.FloatTensor(target_ratings), 
        torch.FloatTensor([float(t) for t in is_terminals])
    )

# Môi trường (Environment) & Memory

Chứa ReplayBuffer, OUNoise và OfflineRecEnv.

In [ ]:
# class ReplayBuffer:
#     def __init__(self, capacity=100000):
#         self.buffer = deque(maxlen=capacity)
    
#     def push(self, state, action, reward, next_state, done):
#         self.buffer.append((state, action, reward, next_state, done))
        
#     def sample(self, batch_size):
#         batch = random.sample(self.buffer, batch_size)
#         state, action, reward, next_state, done = map(np.stack, zip(*batch))
#         return state, action, reward, next_state, done
    
#     def __len__(self): return len(self.buffer)

# class OUNoise: #BỎ
#     def __init__(self, action_dim, mu=0.0, theta=0.15, sigma=0.05, sigma_min=0.05, decay=0.995):
#         self.action_dim = action_dim
#         self.mu = mu
#         self.theta = theta
#         self.sigma = sigma
#         self.sigma_min = sigma_min
#         self.decay = decay
#         self.state = np.ones(action_dim) * mu

#     def reset(self):
#         self.state = np.ones(self.action_dim) * self.mu

#     def decay_sigma(self):
#         self.sigma = max(self.sigma_min, self.sigma * self.decay)

#     def sample(self):
#         dx = self.theta * (self.mu - self.state) + self.sigma * np.random.randn(self.action_dim)
#         self.state = self.state + dx
#         return self.state.copy()

# class OfflineRecEnv: #Lỗi
#     def __init__(self, custom_bert_model, n_negatives=200):
#         self.item_embeddings = custom_bert_model.item_embedding.weight.detach()
#         self.n_items = self.item_embeddings.shape[0]
#         self.n_negatives = n_negatives

#     def step_batch(self, action_matrix, target_item_ids, actual_ratings, device='cuda'):
#         B = action_matrix.shape[0]
#         target_embs = self.item_embeddings[target_item_ids]
#         cos_pos = F.cosine_similarity(action_matrix, target_embs, dim=1)
 
#         neg_ids = torch.randint(1, self.n_items, (B, self.n_negatives), device=device)
#         neg_embs = self.item_embeddings[neg_ids]
 
#         action_expanded = action_matrix.unsqueeze(1).expand(-1, self.n_negatives, -1)
#         cos_neg_all = F.cosine_similarity(action_expanded, neg_embs, dim=2)
#         cos_neg_mean = cos_neg_all.mean(dim=1)
#         cos_neg_std = cos_neg_all.std(dim=1).clamp(min=1e-6)
 
#         direction_signal = (cos_pos - cos_neg_mean) / cos_neg_std
 
#         rating_weight = torch.where(
#             actual_ratings >= 4.0, torch.full_like(actual_ratings, 1.1),
#             torch.where(actual_ratings < 3.0, torch.full_like(actual_ratings, 0.9), torch.ones_like(actual_ratings))
#         )
#         return direction_signal * rating_weight

**sửa**

In [ ]:
# ==========================================================
# REPLAY BUFFER & HÀM REWARD (THAY THẾ CELL 8 & 9 CŨ)
# ==========================================================
import random
from collections import deque
import numpy as np

class ReplayBuffer:
    def __init__(self, capacity=500000):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
        
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = map(np.stack, zip(*batch))
        return state, action, reward, next_state, done
    
    def __len__(self):
        return len(self.buffer)

def get_reward_from_rating(actual_ratings):
    """
    Chuẩn hóa rating từ [1.0, 5.0] về khoảng [-1.0, 1.0]
    Rating 5 ->  1.0  | Rating 4 ->  0.5
    Rating 3 ->  0.0  | Rating 2 -> -0.5 | Rating 1 -> -1.0
    Điều này ép Critic học cách đánh giá giá trị chất lượng của bộ phim.
    """
    return (actual_ratings - 3.0) / 2.0

# Networks (Actor, Critic & State Encoder)

In [ ]:
class StandaloneStateEncoder(nn.Module):
    def __init__(self, custom_bert4rec_model):
        super().__init__()
        self.bert = custom_bert4rec_model
        self.hidden_size = self.bert.hidden_size
        for param in self.bert.parameters():
            param.requires_grad = False

    def get_state(self, user_history_seqs, max_len=200, device='cuda'):
        batch_size = len(user_history_seqs)
        item_seq = torch.zeros((batch_size, max_len), dtype=torch.long, device=device)
        item_seq_len = torch.zeros((batch_size,), dtype=torch.long, device=device)

        for i, seq in enumerate(user_history_seqs):
            seq_len = min(len(seq), max_len)
            item_seq_len[i] = seq_len
            item_seq[i, :seq_len] = torch.tensor(seq[-seq_len:], dtype=torch.long)

        with torch.no_grad():
            prepared_seq = self.bert.reconstruct_test_data(item_seq, item_seq_len)
            seq_output = self.bert.forward(prepared_seq)
            state_vector = gather_indexes(seq_output, item_seq_len - 1)

        return state_vector

class Actor(nn.Module):
    def __init__(self, state_dim, action_dim, beta=0.05):
        super(Actor, self).__init__()
        self.beta = beta
        self.fc1 = nn.Linear(state_dim, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, action_dim) 
        
        nn.init.uniform_(self.fc3.weight, -1e-5, 1e-5)
        nn.init.zeros_(self.fc3.bias)

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        residual = self.fc3(x)
        final_action = F.normalize(state + (self.beta * residual), p=2, dim=-1)
        return final_action

class Critic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(Critic, self).__init__()
        self.fc1 = nn.Linear(state_dim + action_dim, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 1)

    def forward(self, state, action):
        x = torch.cat([state, action], dim=-1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

## TRAINING

In [ ]:
# ==========================================================
# CELL 1: TRAINING OFFLINE RL (TD3 + BC)
# ==========================================================
import torch
import torch.nn.functional as F
import torch.optim as optim
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import numpy as np

def train_rl_only(offline_data, custom_bert, device, epochs=50, batch_size=256, alpha=2.5):
    # Khởi tạo mô hình
    encoder = StandaloneStateEncoder(custom_bert).to(device)
    item_embeddings = encoder.bert.item_embedding.weight.detach()
    
    state_dim = action_dim = encoder.hidden_size
    actor = Actor(state_dim, action_dim, beta=0.5).to(device)
    critic = Critic(state_dim, action_dim).to(device)
    target_actor = Actor(state_dim, action_dim, beta=0.5).to(device)
    target_critic = Critic(state_dim, action_dim).to(device)
    
    target_actor.load_state_dict(actor.state_dict())
    target_critic.load_state_dict(critic.state_dict())
    
    actor_optimizer = optim.Adam(actor.parameters(), lr=1e-4)
    critic_optimizer = optim.Adam(critic.parameters(), lr=1e-4)
    buffer = ReplayBuffer(capacity=500_000)
    
    dataset = OfflineRLDataset(offline_data)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, drop_last=True)
    
    gamma, tau = 0.99, 0.005
    UPDATE_EVERY, CRITIC_WARMUP, Q_CLIP = 10, 4000, 10.0
    total_updates = 0
    history_actor_loss, history_critic_loss = [], []

    print(f"[*] Bắt đầu huấn luyện RL trong {epochs} Epochs...")
    
    for epoch in range(epochs):
        epoch_actor_loss, epoch_critic_loss = 0.0, 0.0
        actor_updates, critic_updates = 0, 0
        
        # Timeline Loading cho Epoch hiện tại
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1:02d}/{epochs}", unit="batch", leave=True)
        
        for step_idx, (histories, target_items, target_ratings, is_terminals) in enumerate(progress_bar):
            # 1. Encode & Xây dựng Batch
            with torch.no_grad():
                s_batch = encoder.get_state(list(histories), device=device)
                next_hists = [(list(h) + [int(t)])[-200:] for h, t in zip(histories, target_items)]
                sn_batch = encoder.get_state(next_hists, device=device)
                s_batch = F.normalize(s_batch, p=2, dim=-1)
                sn_batch = F.normalize(sn_batch, p=2, dim=-1)
                
                t_items_tensor = torch.tensor(target_items, dtype=torch.long, device=device)
                a_real_batch = F.normalize(item_embeddings[t_items_tensor], p=2, dim=-1)
                r_batch = get_reward_from_rating(target_ratings.to(device))
            
            # 2. Push vào Buffer
            for i in range(s_batch.shape[0]):
                buffer.push(
                    s_batch[i].cpu().numpy(), a_real_batch[i].cpu().numpy(),
                    r_batch[i].cpu().item(), sn_batch[i].cpu().numpy(), float(is_terminals[i].item())
                )

            # 3. Cập nhật Actor & Critic
            if len(buffer) > batch_size and step_idx % UPDATE_EVERY == 0:
                s, a, r, s_next, d = [torch.FloatTensor(x).to(device) for x in buffer.sample(batch_size)]
                
                # Critic Update
                with torch.no_grad():
                    q_target_next = target_critic(s_next, target_actor(s_next))
                    q_target = torch.clamp(r.unsqueeze(1) + (1 - d.unsqueeze(1)) * gamma * q_target_next, -Q_CLIP, Q_CLIP)
                
                critic_loss = F.mse_loss(critic(s, a), q_target)
                critic_optimizer.zero_grad()
                critic_loss.backward()
                torch.nn.utils.clip_grad_norm_(critic.parameters(), 0.5)
                critic_optimizer.step()
                
                epoch_critic_loss += critic_loss.item()
                critic_updates += 1
                total_updates += 1
                
                # Actor Update (TD3+BC)
                if total_updates >= CRITIC_WARMUP and critic_updates % 2 == 0:
                    a_pred = actor(s)
                    actor_loss = -critic(s, a_pred).mean() + alpha * F.mse_loss(a_pred, a)
                    
                    actor_optimizer.zero_grad()
                    actor_loss.backward()
                    torch.nn.utils.clip_grad_norm_(actor.parameters(), 0.5)
                    actor_optimizer.step()
                    
                    epoch_actor_loss += actor_loss.item()
                    actor_updates += 1
                    
                    # Soft updates
                    for tp, p in zip(target_actor.parameters(), actor.parameters()): tp.data.copy_(tau * p.data + (1-tau) * tp.data)
                    for tp, p in zip(target_critic.parameters(), critic.parameters()): tp.data.copy_(tau * p.data + (1-tau) * tp.data)
            
            # Cập nhật thông số lên thanh tiến trình
            act_loss_str = f"{epoch_actor_loss/actor_updates:.4f}" if actor_updates > 0 else "Wait..."
            progress_bar.set_postfix({'Critic_L': f"{epoch_critic_loss/max(1, critic_updates):.4f}", 'Actor_L': act_loss_str})

        history_critic_loss.append(epoch_critic_loss / max(1, critic_updates))
        if actor_updates > 0: history_actor_loss.append(epoch_actor_loss / actor_updates)
    
    # Vẽ biểu đồ Loss
    plt.figure(figsize=(10,4))
    plt.plot(history_critic_loss, label='Critic Loss', color='red')
    plt.plot(history_actor_loss, label='Actor Loss', color='blue')
    plt.title("Quá trình Huấn luyện Offline RL")
    plt.legend()
    plt.show()
    
    torch.save(actor.state_dict(), "actor_trained.pth")
    print("[+] Đã lưu mô hình tại 'actor_trained.pth'")
    return actor, encoder

# CHẠY HÀM TRAIN (Bỏ comment để chạy)
# trained_actor, trained_encoder = train_rl_only(offline_data, active_encoder.bert, device, epochs=50)

# Evaluate

**100 NEG**

In [ ]:
# ==========================================================
# CELL 2: ĐÁNH GIÁ THEO PAPER (SAMPLED 100 NEGATIVES)
# ==========================================================
import random

def evaluate_paper_standard(encoder, actor, test_data, device, ks=(1, 5, 10), n_negatives=100, batch_size=128):
    encoder.eval()
    actor.eval()
    item_embeddings = encoder.bert.item_embedding.weight.detach()
    n_items = item_embeddings.shape[0]
    
    metrics = {f'Hit@{k}': 0.0 for k in ks}
    metrics.update({f'NDCG@{k}': 0.0 for k in ks})
    
    print(f"[*] Chấm điểm (Paper Standard): 1 Target vs {n_negatives} Negatives")
    
    # Timeline Loading cho Evaluation
    for i in tqdm(range(0, len(test_data), batch_size), desc="Evaluating (Paper)"):
        batch = test_data[i : i+batch_size]
        histories, targets = [x[0] for x in batch], [x[1] for x in batch]
        
        with torch.no_grad():
            states = encoder.get_state(histories, device=device)
            actions = actor(states) # [B, 64]
            
            for j in range(len(batch)):
                target_id = targets[j]
                
                # Sample 100 negatives
                negs = []
                while len(negs) < n_negatives:
                    neg = random.randint(1, n_items - 1)
                    if neg != target_id: negs.append(neg)
                
                cand_ids = torch.tensor([target_id] + negs, dtype=torch.long, device=device)
                cand_embs = item_embeddings[cand_ids] # [101, 64]
                
                # Tính Cosine & Xếp hạng
                scores = F.cosine_similarity(actions[j].unsqueeze(0), cand_embs, dim=-1)
                rank_list = torch.argsort(scores, descending=True).cpu().numpy()
                target_rank = np.where(rank_list == 0)[0][0]
                
                for k in ks:
                    if target_rank < k:
                        metrics[f'Hit@{k}'] += 1.0
                        metrics[f'NDCG@{k}'] += 1.0 / np.log2(target_rank + 2)
                        
    for key in metrics: metrics[key] /= len(test_data)
        
    print("-" * 50)
    for k in ks:
        print(f"Hit@{k:<2}: {metrics[f'Hit@{k}']:.4f}  |  NDCG@{k:<2}: {metrics[f'NDCG@{k}']:.4f}")
    print("-" * 50)
    return metrics

# CHẠY HÀM ĐÁNH GIÁ (Bỏ comment để chạy)
metrics_paper = evaluate_paper_standard(trained_encoder, trained_actor, test_data, device)

**Full-Rank Evaluation**

In [ ]:
# ==========================================================
# CELL 3: ĐÁNH GIÁ FULL-RANK (SO SÁNH VỚI TOÀN BỘ ITEMS)
# ==========================================================
def evaluate_full_rank(encoder, actor, test_data, device, ks=(1, 5, 10, 20), batch_size=128):
    encoder.eval()
    actor.eval()
    
    # Bảng Embedding của TOÀN BỘ items (3707 x 64)
    all_item_embeddings = encoder.bert.item_embedding.weight.detach() 
    
    metrics = {f'Hit@{k}': 0.0 for k in ks}
    metrics.update({f'NDCG@{k}': 0.0 for k in ks})
    
    print(f"[*] Chấm điểm (Full-Rank): Xếp hạng trên toàn bộ {all_item_embeddings.shape[0]} bộ phim")
    
    # Timeline Loading cho Full-rank Evaluation
    for i in tqdm(range(0, len(test_data), batch_size), desc="Evaluating (Full-Rank)"):
        batch = test_data[i : i+batch_size]
        histories, targets = [x[0] for x in batch], [x[1] for x in batch]
        
        with torch.no_grad():
            states = encoder.get_state(histories, device=device)
            actions = actor(states) # [B, 64]
            
            # Tính Cosine Similarity một lần cho tất cả Batch với TOÀN BỘ Items
            # Tối ưu hóa GPU matrix multiplication: [B, 64] x [64, N_items]
            scores = torch.matmul(actions, all_item_embeddings.T) # -> [B, N_items]
            
            # Tính chuẩn hóa thủ công (Cosine Similarity = A*B / (|A|*|B|))
            a_norm = actions.norm(p=2, dim=1, keepdim=True)
            b_norm = all_item_embeddings.norm(p=2, dim=1, keepdim=True).T
            scores = scores / (a_norm * b_norm).clamp(min=1e-8)
            
            # Sắp xếp xếp hạng cho cả Batch
            _, sorted_indices = torch.sort(scores, dim=-1, descending=True)
            
            for j in range(len(batch)):
                target_id = targets[j]
                
                # Tìm xem target_id nằm ở vị trí thứ mấy trong mảng đã sort
                rank_list = sorted_indices[j].cpu().numpy()
                target_rank = np.where(rank_list == target_id)[0][0]
                
                for k in ks:
                    if target_rank < k:
                        metrics[f'Hit@{k}'] += 1.0
                        metrics[f'NDCG@{k}'] += 1.0 / np.log2(target_rank + 2)

    for key in metrics: metrics[key] /= len(test_data)
        
    print("=" * 60)
    print("KẾT QUẢ ĐÁNH GIÁ FULL-RANK (Thực tế khắt khe nhất)")
    print("=" * 60)
    for k in ks:
        print(f"Hit@{k:<2}: {metrics[f'Hit@{k}']:.4f}  |  NDCG@{k:<2}: {metrics[f'NDCG@{k}']:.4f}")
    print("=" * 60)
    return metrics

# CHẠY HÀM ĐÁNH GIÁ FULL RANK (Bỏ comment để chạy)
metrics_full = evaluate_full_rank(trained_encoder, trained_actor, test_data, device)

# TEST

In [ ]:
# # ==========================================================
# # CELL: HÀM CHUẨN BỊ DỮ LIỆU TEST (Fixed)
# # ==========================================================
# def prepare_rl_test_data(rating_file_path, max_seq_len=200):
#     print(f"[*] Đang chuẩn bị dữ liệu test từ {rating_file_path}...")
    
#     # 1. Load và Sort dữ liệu
#     ratings_cols = ['UserID', 'MovieID', 'Rating', 'Timestamp']
#     ratings = pd.read_csv(rating_file_path, sep='::', engine='python', names=ratings_cols)
#     ratings = ratings.sort_values(by=['UserID', 'Timestamp']).reset_index(drop=True)
    
#     # 2. Map MovieID về index 1..N
#     raw_movie_ids = sorted(ratings['MovieID'].unique())
#     movie2id = {raw_id: i + 1 for i, raw_id in enumerate(raw_movie_ids)}
#     ratings['Movie_Encoded'] = ratings['MovieID'].map(movie2id)
    
#     # 3. Gom nhóm theo User
#     user_sequences = ratings.groupby('UserID')['Movie_Encoded'].apply(list).to_dict()
    
#     test_data = []
#     for user_id, seq in user_sequences.items():
#         if len(seq) < 5: continue # Bỏ qua user quá ít lịch sử
        
#         # Lấy bộ phim cuối cùng làm Target, lịch sử trước đó là History
#         target_item = seq[-1]
#         history = seq[-max_seq_len:-1] # Lấy tối đa max_seq_len phim trước đó
        
#         test_data.append((history, target_item))
    
#     print(f"[+] Đã tạo thành công {len(test_data)} samples cho tập Test.")
#     return test_data

# # --- Sau khi chạy xong hàm này, biến 'test_data' đã tồn tại ---
# test_data = prepare_rl_test_data(RATING_FILE_PATH, max_seq_len=200)

In [ ]:
# # ==========================================================
# # CELL KHỞI TẠO BIẾN (Chạy Cell này trước khi test)
# # ==========================================================

# # 1. Xác định thiết bị
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # 2. Khởi tạo và nạp Encoder (BERT4Rec đã train)
# # Lưu ý: Đảm bảo bạn đã có hàm khởi tạo model BERT4Rec của bạn
# active_encoder = StandaloneStateEncoder(BERT4Rec).to(device) 

# # 3. Chuẩn bị dữ liệu Test
# # Dùng lại hàm chuẩn bị dữ liệu cũ mà bạn đã viết để tạo biến 'test_data'
# test_data = prepare_rl_test_data(RATING_FILE_PATH, max_seq_len=200)

# # 4. Nạp model Actor đã train (Từ Cell Training)
# # Đảm bảo file này đã tồn tại sau khi bạn chạy Cell 1 (Train)
# trained_actor = Actor(state_dim=active_encoder.hidden_size, action_dim=active_encoder.hidden_size, beta=0.5).to(device)
# trained_actor.load_state_dict(torch.load("actor_trained.pth", map_location=device))
# trained_actor.eval()

# print("[+] Đã khởi tạo xong các biến: active_encoder, test_data, trained_actor")

In [ ]:
# # ==========================================================
# # CELL 5: INFERENCE (DỰ ĐOÁN THỰC TẾ CHO 1 USER)
# # ==========================================================
# def recommend_for_user(encoder, actor, user_history, device, top_k=10):
#     """
#     Nhận vào lịch sử xem phim của 1 User và trả về Top K bộ phim gợi ý.
#     """
#     encoder.eval()
#     actor.eval()
    
#     with torch.no_grad():
#         # 1. Trích xuất State hiện tại của User
#         state = encoder.get_state([user_history], device=device) # [1, 64]
        
#         # 2. RL Actor đưa ra vector hành động (sở thích phim tiếp theo)
#         action_vector = actor(state) # [1, 64]
        
#         # 3. So sánh vector này với toàn bộ kho phim
#         all_item_embeddings = encoder.bert.item_embedding.weight.detach()
#         scores = F.cosine_similarity(action_vector, all_item_embeddings, dim=-1) # [3707]
        
#         # 4. Loại bỏ những phim User đã xem rồi (không gợi ý lại)
#         history_tensor = torch.tensor(user_history, dtype=torch.long, device=device)
#         scores[history_tensor] = -1e9 
        
#         # 5. Lấy Top K phim có điểm Cosine cao nhất
#         top_scores, top_indices = torch.topk(scores, k=top_k)
        
#     return top_indices.cpu().numpy(), top_scores.cpu().numpy()

# # -------- CHẠY THỬ --------
# # Giả sử lịch sử xem phim của User là các ID: 1, 50, 231, 100
# sample_history = [1, 50, 231, 100]

# recommended_ids, confidence_scores = recommend_for_user(
#     encoder=active_encoder,
#     actor=test_actor,
#     user_history=sample_history,
#     device=device,
#     top_k=5
# )

# print(f"Lịch sử xem phim (IDs): {sample_history}")
# print("-" * 40)
# print("Hệ thống RL gợi ý 5 bộ phim tiếp theo:")
# for rank, (item_id, score) in enumerate(zip(recommended_ids, confidence_scores)):
#     print(f"Top {rank+1}: Movie ID {item_id:4d} (Độ tự tin: {score:.4f})")